<img src="https://raw.githubusercontent.com/asterisk-labs/rumi/main/img/rumi-lockup-tight.svg" width="440">

# rumi in ten minutes

**rumi** is a predictable raster format for AI training data. Pixels stay in a
fixed BigTIFF-derived container that is deliberately not TIFF or GeoTIFF.
Alongside it rumi keeps a small binary header
that says where every frame lives, so a reader jumps straight to the frames it
wants without parsing the IFD.

Frames are compressed with [OpenZL](https://github.com/facebook/openzl)
through [geozl](https://github.com/asterisk-labs/geozl), which picks a codec
graph that fits the data instead of one general purpose compressor.

This notebook builds a scene, cuts it into frames, compresses it, writes it, and reads it
back. Run the cells top to bottom.

- Repo <https://github.com/asterisk-labs/rumi>
- Spec <https://github.com/asterisk-labs/rumi/blob/main/SPEC.md>

## Setup

geozl ships wheels for Linux x86_64 and macOS arm64. Colab is Linux x86_64, so
this works out of the box. rasterio is only here to make and read the source
image, rumi never needs it.

In [ ]:
!pip install "rumi-eo[write]"

In [ ]:
import numpy as np
import rasterio
import geozl
import rumi

rumi.__version__, geozl.__version__

## A scene to work with

Three bands at 5490 px, the size of a Sentinel-2 20 m tile, in UTM 18S over
Peru. Swap this cell for your own file if you have one.

We also write the same pixels as a tiled DEFLATE GeoTIFF, which is the honest
baseline to compare against later. Comparing to an uncompressed file would
flatter rumi for no reason.

In [ ]:
from rasterio.transform import from_origin

N = 5490
rng = np.random.default_rng(0)
y, x = np.mgrid[0:N, 0:N].astype(np.float32)

bands = np.empty((3, N, N), np.uint16)
for i, f in enumerate((
        lambda: 2000 + 800 * np.sin(x / 90) * np.cos(y / 120),
        lambda: 1500 + 600 * np.cos(x / 70),
        lambda: 3000 + 900 * np.sin((x + y) / 110))):
    band = f() + rng.normal(0, 40, (N, N)).astype(np.float32)
    bands[i] = band.clip(0, 10000).astype(np.uint16)
del y, x, band

common = dict(driver="GTiff", height=N, width=N, count=3, dtype="uint16",
              crs="EPSG:32718", transform=from_origin(500000, 8000000, 10, 10))

with rasterio.open("demo.tif", "w", **common) as dst:
    dst.write(bands)

with rasterio.open("demo_deflate.tif", "w", **common, tiled=True,
                   blockxsize=512, blockysize=512,
                   compress="deflate", predictor=2) as dst:
    dst.write(bands)

src = rasterio.open("demo.tif")
src.profile

## The data model

<img src="https://raw.githubusercontent.com/asterisk-labs/rumi/main/img/rumi-data-model.svg" width="620">

A **FrameTable** is the grid in wire order, with one row per frame. A tile frame
holds one band; a cell frame holds all bands. Edge tiles are **cut, never padded**,
so a tile at the right edge is genuinely narrower; nothing is invented or decoded
never written.

In [ ]:
tf = rumi.frames(src.read(), 512)
tf

`unit='tile'` creates one 2D frame per band and grid position. `unit='cell'`
creates one 3D frame with all bands at a position. Band, row and column remain
available as dimensions for routing frames to compression graphs.

In [ ]:
tf.columns, tf.dims

In [ ]:
t = tf[0]
t.tile, t.cell, t.data.shape, t.compressed

5490 is not a multiple of 512, so the last row and column come up short.
Four distinct shapes for any image, and that matters in a moment.

In [ ]:
sorted({t.data.shape for t in tf})

## Compressing

rumi does not compress. You pick the graph, which is the point, since a codec
that suits a DEM is not the one that suits an S1 backscatter raster.
`geozl.profile` measures the candidates on a real tile.

In [ ]:
import pandas as pd
pd.DataFrame(geozl.profile(tf[0].data)).head(6)

One catch worth knowing before it bites you. A graph carries the stride it
was built with, and edge tiles are shorter, so a graph built on a 512x512 tile
refuses the 370x370 corner. Keep one graph per tile shape, at most four.

In [ ]:
best = geozl.profile(tf[0].data)[0]["graph"]

for t in tf:
    graph = geozl.graph(t.data, best)
    t.compressed = geozl.compress(t.data, graph=graph)
tf

## Writing

<img src="https://raw.githubusercontent.com/asterisk-labs/rumi/main/img/rumi-index.svg" width="700">

`rumi.write` returns the path and the **header**, a compact sidecar of raw
bytes. It is not a competing format: it describes the rumi container itself,
which is deliberately not TIFF or GeoTIFF.
The header just means a reader never has to open the file to know what is in it,
or walk the IFD to find a frame.

In [ ]:
path, header = rumi.write("demo.rumi", tf,
                          transform=src.transform,
                          crs=src.crs.to_epsg())

import os
raw = os.path.getsize("demo.tif")
for label, name in (("uncompressed", "demo.tif"),
                    ("GeoTIFF DEFLATE", "demo_deflate.tif"),
                    ("rumi", path)):
    n = os.path.getsize(name)
    print(f"{label:18} {n:>12,} bytes   {raw / n:4.2f}x")

print(f"\n{'header':18} {len(header):>12,} bytes")

`transform` takes rasterio's `Affine` order, `(x_res, row_rot, x_origin,
col_rot, y_res, y_origin)`. Passing `src.transform.to_gdal()` instead writes a
georeferencing that is silently wrong, so hand it the `Affine` itself.

## The header on its own

Bytes in, answers out. No file is opened here.

In [ ]:
h = rumi.RumiHeader(header)
h

In [ ]:
h.to_dict()

## Reading

`read` takes the header bytes you cached. Leave it out and rumi reads it off the
file, which costs one extra open.

In [ ]:
full = rumi.read(path, header, num_threads=4)
full.shape, full.dtype, np.array_equal(full, src.read())

A window and a band subset only decode the frames they touch. Compare the
wall time against the full read above.

In [ ]:
%%time
patch = rumi.read(path, header, b=[0, 2], y=(2000, 2512), x=(3000, 3512))
patch.shape

`pattern` names the axis order you want out, so the transpose happens
during assembly rather than as a copy afterwards.

In [ ]:
rumi.read(path, header, pattern="y x b", y=(0, 256), x=(0, 256)).shape

A list of paths reads a stack, with `n` as the image axis.

In [ ]:
cube = rumi.read([path, path, path], [header] * 3,
                 pattern="n b y x", y=(0, 512), x=(0, 512))
cube.shape

`framework` picks what comes back. `"numpy"` is the default, `"torch"`,
`"jax"` and `"tensorflow"` work the same way, and `None` hands you a zero copy
DLPack array that any of them can adopt.

In [ ]:
arr = rumi.read(path, header, framework=None)
print(arr)

np.from_dlpack(rumi.read(path, header, framework=None)).shape

## Why the header is bytes

<img src="https://raw.githubusercontent.com/asterisk-labs/rumi/main/img/rumi-catalog.svg" width="700">

Because bytes go in a column. Put the header in Parquet next to the path and a
loader knows the shape, dtype and tile grid of every scene in the dataset
without touching object storage.

## Where to go next

- The spec, if you want to write a reader
  <https://github.com/asterisk-labs/rumi/blob/main/SPEC.md>
- geozl, the codec side
  <https://github.com/asterisk-labs/geozl>
- Issues and questions
  <https://github.com/asterisk-labs/rumi/issues>

<br>

<img src="https://raw.githubusercontent.com/asterisk-labs/rumi/main/img/asterisk_banner.svg" width="200">